# ⚡ TurboQuant Algorithm Deep Dive
### Notebook 01 — Interactive Math Walkthrough · CPU Only

**References:**
- TurboQuant: [arXiv:2504.19874](https://arxiv.org/abs/2504.19874) (ICLR 2026)
- QJL: [arXiv:2406.03482](https://arxiv.org/abs/2406.03482)
- PolarQuant: [arXiv:2502.02617](https://arxiv.org/abs/2502.02617)

---
This notebook walks through the math of all three algorithms from first principles:
1. **QJL** — 1-bit inner product estimation via Johnson-Lindenstrauss
2. **PolarQuant** — Polar coordinate quantizer with Haar rotation
3. **TurboQuant** — Two-stage combination for near-optimal distortion

> ✅ **No GPU required** — everything runs on CPU in under 2 minutes.

In [ ]:
# @title 🔍 Environment Check
import sys, torch, numpy as np
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} (not needed for this notebook)')

In [ ]:
# @title 📦 Install Dependencies (run once)
# %%capture
# !pip install torch numpy scipy matplotlib seaborn
# !pip install -e /content/TQ-infer-engine   # if running in Colab

In [ ]:
# @title 📚 Imports & Global Seed
import math, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Try importing from the installed tqe package
try:
    import sys, os
    sys.path.insert(0, os.path.join(os.getcwd(), '..'))
    from tqe.algorithms.qjl import QJLQuantizer
    from tqe.algorithms.polar_quant import PolarQuantizer
    from tqe.algorithms.turbo_quant import TurboQuantizer
    print('✅ tqe package imported successfully')
except ImportError as e:
    print(f'⚠️  Could not import tqe: {e}')
    print('   Run: pip install -e .. from the notebooks/ directory')

torch.manual_seed(42)
np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.titlesize': 14})
DEVICE = 'cpu'
print(f'Device: {DEVICE}')

---
## Part 1 · QJL — Quantized Johnson-Lindenstrauss

### Mathematical Foundation

Given vectors $k, q \in \mathbb{R}^d$, QJL estimates $\langle k, q \rangle$ using:

$$\hat{\langle k, q \rangle} = \frac{2}{\pi} \cdot \frac{d}{m} \sum_{i=1}^{m} (\Phi q)_i \cdot \text{sign}(\Phi k)_i$$

where $\Phi \in \mathbb{R}^{m \times d}$ has i.i.d. $\mathcal{N}(0, 1/m)$ entries.

**Key properties:**
- **Unbiased**: $\mathbb{E}[\hat{\langle k, q \rangle}] = \langle k, q \rangle$
- **Zero overhead**: sign bits need no quantization constants
- The $\frac{2}{\pi}$ constant comes from $\mathbb{E}[\text{sign}(x) \cdot y] = \frac{2}{\pi} \cdot \mathbb{E}[|x|] \cdot \cos(\angle(x,y))$

In [ ]:
# @title Cell 1.1 — Encode with QJL: shape and dtype check
d, m = 256, 256
n    = 200
qjl  = QJLQuantizer(input_dim=d, proj_dim=m, seed=42)

torch.manual_seed(0)
vectors = torch.randn(n, d)
codes   = qjl.encode(vectors)

print(f'Input  shape : {list(vectors.shape)}  dtype={vectors.dtype}')
print(f'Codes  shape : {list(codes.shape)}   dtype={codes.dtype}')
print(f'Unique values: {codes.unique().tolist()}  (must be ±1 only)')
print(f'Memory: {qjl.memory_bytes(n):,} bytes  vs  {n*d*4:,} bytes (FP32 original)')
print(f'Ratio : {n*d*4 / qjl.memory_bytes(n):.1f}× (just the sign bits)')

In [ ]:
# @title Cell 1.2 — Inner product estimation: scatter plot vs truth
torch.manual_seed(42)
n_pairs = 1000
keys    = torch.randn(n_pairs, d)
queries = torch.randn(n_pairs, d)

true_ips  = (keys * queries).sum(-1).numpy()
key_codes = qjl.encode(keys)
est_ips   = qjl.estimate_inner_product(queries, key_codes).numpy()

corr = float(np.corrcoef(true_ips, est_ips)[0, 1])
rel_err = float(np.mean(np.abs(est_ips - true_ips) / (np.abs(true_ips).clip(0.1))))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
axes[0].scatter(true_ips, est_ips, alpha=0.3, s=8, color='#2E86AB')
lim = max(abs(true_ips).max(), abs(est_ips).max()) * 1.05
axes[0].plot([-lim, lim], [-lim, lim], 'r--', linewidth=1.5, label='Perfect')
axes[0].set_xlabel('True ⟨k, q⟩')
axes[0].set_ylabel('QJL Estimated ⟨k, q⟩')
axes[0].set_title(f'QJL Inner Product Estimation\n(r={corr:.4f}, rel_err={rel_err:.1%})')
axes[0].legend()

# Error histogram
errors = est_ips - true_ips
axes[1].hist(errors, bins=60, color='#A23B72', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='red', linewidth=2, linestyle='--', label=f'Mean={errors.mean():.4f}')
axes[1].set_xlabel('Estimation Error')
axes[1].set_ylabel('Count')
axes[1].set_title('Error Distribution (should be zero-mean)')
axes[1].legend()

plt.tight_layout()
plt.savefig('qjl_inner_product.png', dpi=150)
plt.show()
print(f'Correlation: {corr:.5f}  |  Mean relative error: {rel_err:.2%}')

In [ ]:
# @title Cell 1.3 — Effect of projection dim m on estimation error
torch.manual_seed(1)
proj_dims = [32, 64, 128, 256, 512]
d = 256
n_test = 500
keys    = torch.randn(n_test, d)
queries = torch.randn(n_test, d)
true_ips = (keys * queries).sum(-1).numpy()

mean_rel_errors, std_rel_errors = [], []
for m in proj_dims:
    qjl_m = QJLQuantizer(input_dim=d, proj_dim=m, seed=42)
    codes  = qjl_m.encode(keys)
    est    = qjl_m.estimate_inner_product(queries, codes).numpy()
    rel_e  = np.abs(est - true_ips) / np.abs(true_ips).clip(0.1)
    mean_rel_errors.append(rel_e.mean())
    std_rel_errors.append(rel_e.std())

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(proj_dims, mean_rel_errors, yerr=std_rel_errors,
            fmt='o-', color='#2E86AB', linewidth=2, markersize=8, capsize=5)
ax.axhline(0.05, color='red', linestyle='--', alpha=0.7, label='5% threshold')
ax.set_xlabel('Projection Dim m')
ax.set_ylabel('Mean Relative Error')
ax.set_title('QJL: Estimation Error vs Projection Dimension (d=256)')
ax.legend()
ax.set_xticks(proj_dims)
plt.tight_layout()
plt.savefig('qjl_proj_dim.png', dpi=150)
plt.show()

print('Projection dim → Mean Rel Error:')
for m, e in zip(proj_dims, mean_rel_errors):
    print(f'  m={m:4d}: {e:.3%}')

---
## Part 2 · PolarQuant — Polar Coordinate Quantizer

### Mathematical Foundation

After applying a **Haar random rotation** $R$, group each pair $(v_{2i}, v_{2i+1})$:

$$r_i = \|(v_{2i}, v_{2i+1})\|_2 \qquad \theta_i = \text{atan2}(v_{2i+1}, v_{2i}) \in [-\pi, \pi]$$

**Why random rotation?** → After $R$, the angles $\theta_i$ distribute **near-uniformly** on $[-\pi, \pi]$, making a uniform scalar quantizer (no adaptive codebook!) near-optimal.

**Bit budget** at 4 bits/dim:
- 3 bits for angle $\theta_i$ (2³ = 8 levels over $[-\pi, \pi]$)
- 1 bit for radius $r_i$ (2 levels after max-normalization)
- Total: 4 bits per 2 values = **2 bits/dim** ✓

In [ ]:
# @title Cell 2.1 — Angle distribution before/after random rotation
torch.manual_seed(7)
d = 128
n = 5000
pq = PolarQuantizer(input_dim=d, bits_per_dim=4.0, rotation_seed=42)

v = torch.randn(n, d)  # Gaussian vectors

# Before rotation: angles from raw pairs
pairs_raw = v.reshape(n, d//2, 2)
angles_before = torch.atan2(pairs_raw[..., 1], pairs_raw[..., 0]).reshape(-1).numpy()

# After rotation
rotated = v @ pq.R.T
pairs_rot = rotated.reshape(n, d//2, 2)
angles_after = torch.atan2(pairs_rot[..., 1], pairs_rot[..., 0]).reshape(-1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), subplot_kw={'projection': 'polar'})

for ax, angles, title, color in zip(
    axes,
    [angles_before, angles_after],
    ['Before Rotation (non-uniform)', 'After Haar Rotation (near-uniform)'],
    ['#C73E1D', '#2E86AB']
):
    ax.hist(angles, bins=72, color=color, alpha=0.8)
    ax.set_title(title, pad=20)

plt.suptitle('Angle Distribution: PolarQuant Haar Rotation Effect', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('pq_angle_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Uniformity test: std of angle histogram counts
hist_before = np.histogram(angles_before, bins=36)[0].astype(float)
hist_after  = np.histogram(angles_after,  bins=36)[0].astype(float)
cv_before = hist_before.std() / hist_before.mean()
cv_after  = hist_after.std()  / hist_after.mean()
print(f'Coefficient of Variation (lower=more uniform):')
print(f'  Before rotation: {cv_before:.4f}')
print(f'  After  rotation: {cv_after:.4f}  (should be smaller)')

In [ ]:
# @title Cell 2.2 — Encode/decode roundtrip at 2, 4, 8 bits
torch.manual_seed(42)
d = 64
n = 100
v = torch.randn(n, d)
bits_list = [2.0, 3.0, 4.0, 8.0]

fig, axes = plt.subplots(1, len(bits_list), figsize=(16, 4))
mse_results = {}

for ax, bits in zip(axes, bits_list):
    pq    = PolarQuantizer(input_dim=d, bits_per_dim=bits, rotation_seed=42)
    codes = pq.encode(v)
    v_hat = pq.decode(codes)
    mse   = float(((v - v_hat)**2).mean())
    var   = float(v.var())
    mse_results[bits] = (mse, mse/var)

    # Show first 5 vectors
    for i in range(5):
        ax.plot(v[i].numpy(), color='#2E86AB', alpha=0.6, linewidth=1)
        ax.plot(v_hat[i].numpy(), color='#C73E1D', alpha=0.6, linewidth=1,
                linestyle='--')

    ax.set_title(f'{bits:.0f}-bit PolarQuant\nMSE/σ²={mse/var:.4f}')
    ax.set_xlabel('Dimension')
    if ax == axes[0]:
        ax.set_ylabel('Value')

# Legend patch
from matplotlib.lines import Line2D
handles = [
    Line2D([0],[0], color='#2E86AB', label='Original'),
    Line2D([0],[0], color='#C73E1D', linestyle='--', label='Reconstructed'),
]
axes[-1].legend(handles=handles, loc='upper right')
plt.suptitle('PolarQuant Encode/Decode Roundtrip at Multiple Bit-widths', fontsize=13)
plt.tight_layout()
plt.savefig('pq_roundtrip.png', dpi=150)
plt.show()

print(f'{'Bits':>6} | {'MSE':>10} | {'MSE/σ²':>10}')
print('-' * 32)
for bits, (mse, rel) in mse_results.items():
    print(f'{bits:>6.1f} | {mse:>10.6f} | {rel:>10.6f}')

In [ ]:
# @title Cell 2.3 — MSE vs bit-width bar chart
from tqe.utils.math_utils import theoretical_distortion

torch.manual_seed(99)
d, n = 256, 2000
v = torch.randn(n, d)
sigma_sq = float(v.var())
bits_range = [2.0, 3.0, 4.0, 6.0, 8.0]

pq_mses, theory_mses = [], []
for bits in bits_range:
    pq    = PolarQuantizer(input_dim=d, bits_per_dim=bits, rotation_seed=42)
    codes = pq.encode(v)
    v_hat = pq.decode(codes)
    pq_mses.append(float(((v - v_hat)**2).mean()))
    theory_mses.append(theoretical_distortion(sigma_sq, bits))

x = np.arange(len(bits_range))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, pq_mses,     w, label='PolarQuant',       color='#A23B72', alpha=0.9)
bars2 = ax.bar(x + w/2, theory_mses, w, label='Theoretical D*(R)', color='#3B1F2B', alpha=0.7)

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
            f'{bar.get_height():.4f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([f'{b:.0f} bit' for b in bits_range])
ax.set_yscale('log')
ax.set_ylabel('MSE Distortion (log scale)')
ax.set_title('PolarQuant MSE vs Theoretical Shannon Bound (d=256, n=2000)')
ax.legend()
plt.tight_layout()
plt.savefig('pq_mse_vs_bits.png', dpi=150)
plt.show()

---
## Part 3 · TurboQuant — Full Two-Stage Pipeline

### Mathematical Foundation

**Total MSE** of TurboQuant:
$$\mathbb{E}[\|v - \hat{v}_{TQ}\|^2] \approx \mathbb{E}[\|e_{polar}\|^2] - \mathbb{E}[\|\hat{e}_{qjl}\|^2]$$

where $e_{polar} = v - \text{PolarQuant}(v)$ is the PolarQuant residual and $\hat{e}_{qjl}$ is QJL's approximation of the residual.

**Inner product estimation** (the core attention operation):
$$\langle v_k, q \rangle \approx \underbrace{\langle \hat{v}_{pq,k},\, q \rangle}_{\text{PolarQuant}} + \underbrace{\frac{2}{\pi}\frac{d}{m}\langle \Phi q,\, \text{codes}_{qjl,k} \rangle}_{\text{QJL correction}}$$

**Key result**: TurboQuant achieves distortion within **2.7×** of the Shannon rate-distortion bound.

In [ ]:
# @title Cell 3.1 — Encode 1000 vectors and compare MSE: TurboQuant vs PolarQuant vs QJL
torch.manual_seed(42)
d, n = 256, 1000
v = torch.randn(n, d)
sigma_sq = float(v.var())

results = {}
for bits in [2.0, 3.0, 4.0, 6.0, 8.0]:
    # --- TurboQuant ---
    tq     = TurboQuantizer(d, total_bits_per_dim=bits)
    tq_codes = tq.encode(v)
    tq_hat   = tq.decode(tq_codes)
    tq_mse   = float(((v - tq_hat)**2).mean())

    # --- PolarQuant (bits-1 to fair compare) ---
    pq       = PolarQuantizer(d, bits_per_dim=bits, rotation_seed=42)
    pq_codes = pq.encode(v)
    pq_hat   = pq.decode(pq_codes)
    pq_mse   = float(((v - pq_hat)**2).mean())

    from tqe.utils.math_utils import theoretical_distortion
    theory = theoretical_distortion(sigma_sq, bits)

    results[bits] = {'TurboQuant': tq_mse, 'PolarQuant': pq_mse, 'Theory': theory}

# QJL baseline (1-bit)
qjl = QJLQuantizer(d, d, seed=42)
qjl_hat = qjl.decode_approximate(qjl.encode(v))
qjl_mse = float(((v - qjl_hat)**2).mean())

bits_arr  = sorted(results.keys())
tq_mses   = [results[b]['TurboQuant'] for b in bits_arr]
pq_mses   = [results[b]['PolarQuant'] for b in bits_arr]
th_mses   = [results[b]['Theory']     for b in bits_arr]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(bits_arr, tq_mses, 'o-', color='#2E86AB', linewidth=2.5, markersize=9, label='TurboQuant')
ax.plot(bits_arr, pq_mses, 's--', color='#A23B72', linewidth=2, markersize=8, label='PolarQuant')
ax.plot(bits_arr, th_mses, '^:', color='#3B1F2B', linewidth=2, markersize=8, label='Theoretical D*(R)')
ax.axhline(qjl_mse, color='#F18F01', linestyle='-.',
           linewidth=2, label=f'QJL (1-bit) = {qjl_mse:.4f}')
ax.set_xlabel('Bits per Dimension')
ax.set_ylabel('MSE Distortion')
ax.set_yscale('log')
ax.set_title('MSE Distortion vs Bit-width: TurboQuant vs Baselines (d=256, n=1000)')
ax.legend()
ax.set_xticks(bits_arr)
plt.tight_layout()
plt.savefig('turbo_distortion_comparison.png', dpi=150)
plt.show()

print(f'{'Bits':>6} | {'TurboQuant':>12} | {'PolarQuant':>12} | {'Theory':>12} | {'TQ/Theory':>10}')
print('-' * 65)
for bits in bits_arr:
    tq_m = results[bits]['TurboQuant']
    pq_m = results[bits]['PolarQuant']
    th_m = results[bits]['Theory']
    ratio = tq_m / th_m
    print(f'{bits:>6.1f} | {tq_m:>12.6f} | {pq_m:>12.6f} | {th_m:>12.6f} | {ratio:>10.2f}×')

In [ ]:
# @title Cell 3.2 — Inner product estimation quality: violin plot of relative errors
torch.manual_seed(5)
d, n = 256, 500
keys    = torch.randn(n, d)
queries = torch.randn(n, d)
true_ips = (keys * queries).sum(-1).numpy()
mask = np.abs(true_ips) > 0.1  # avoid near-zero division

violin_data  = {}
bits_to_test = [2.0, 3.0, 4.0, 8.0]

for bits in bits_to_test:
    tq     = TurboQuantizer(d, total_bits_per_dim=bits)
    codes  = tq.encode(keys)
    est    = tq.estimate_inner_products(queries, codes).numpy()
    rel_e  = np.abs(est[mask] - true_ips[mask]) / np.abs(true_ips[mask])
    violin_data[f'{bits:.0f}bit TQ'] = rel_e

# Add QJL baseline
qjl = QJLQuantizer(d, d, seed=42)
key_codes = qjl.encode(keys)
est_qjl   = qjl.estimate_inner_product(queries, key_codes).numpy()
violin_data['QJL (1bit)'] = np.abs(est_qjl[mask] - true_ips[mask]) / np.abs(true_ips[mask])

fig, ax = plt.subplots(figsize=(11, 5))
labels = list(violin_data.keys())
data   = [violin_data[k] for k in labels]
vp = ax.violinplot(data, positions=range(len(labels)), showmedians=True)
colors = ['#C73E1D', '#A23B72', '#2E86AB', '#44BBA4', '#F18F01']
for i, (body, color) in enumerate(zip(vp['bodies'], colors)):
    body.set_facecolor(color)
    body.set_alpha(0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel('Relative Error |est - true| / |true|')
ax.set_title('TurboQuant Inner Product Estimation Error Distribution (d=256, n=500)')
ax.axhline(0.05, color='red', linestyle='--', alpha=0.6, label='5% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('turbo_ip_violin.png', dpi=150)
plt.show()

In [ ]:
# @title Cell 3.3 — Compression ratio table (all bit-widths vs FP16, FP32)
print('=' * 70)
print(f'{'Method':20} {'Bits':>6} {'FP32 ratio':>12} {'FP16 ratio':>12} {'Mem/1K vec':>12}')
print('-' * 70)

for bits in [2.0, 3.0, 4.0, 6.0, 8.0]:
    tq = TurboQuantizer(input_dim=256, total_bits_per_dim=bits)
    ratio_fp32 = tq.compression_ratio(1000, original_dtype_bytes=4)
    ratio_fp16 = tq.compression_ratio(1000, original_dtype_bytes=2)
    mem_kb     = tq.memory_bytes(1000) / 1024
    print(f'{'TurboQuant':20} {bits:>6.1f} {ratio_fp32:>11.2f}× {ratio_fp16:>11.2f}× {mem_kb:>10.1f} KB')

print('-' * 70)
fp32_kb = 1000 * 256 * 4 / 1024
fp16_kb = 1000 * 256 * 2 / 1024
print(f'{'Baseline FP32':20} {'32':>6} {'1.00':>12} {'0.50':>12} {fp32_kb:>10.1f} KB')
print(f'{'Baseline FP16':20} {'16':>6} {'2.00':>12} {'1.00':>12} {fp16_kb:>10.1f} KB')
print('=' * 70)

In [ ]:
# @title Cell 4 — 📋 Conclusion: All Key Metrics Summary
torch.manual_seed(42)
d, n = 256, 1000
v       = torch.randn(n, d)
q       = torch.randn(n, d)
true_ips = (v * q).sum(-1)
sigma_sq = float(v.var())

print('\n' + '='*75)
print('  TurboQuant Algorithm Deep Dive — KEY RESULTS SUMMARY')
print('  d=256, n=1000 random FP32 vectors')
print('='*75)

print(f'\n{'Method':20} {'Bits':>5} {'MSE/σ²':>10} {'IP RelErr':>10} {'Mem/vec':>10} {'Ratio':>8}')
print('-'*70)

# QJL
qjl = QJLQuantizer(d, d, seed=42)
qjl_hat = qjl.decode_approximate(qjl.encode(v))
qjl_mse = float(((v - qjl_hat)**2).mean()) / sigma_sq
qjl_ip_est = qjl.estimate_inner_product(q, qjl.encode(v))
qjl_ip_err = float((qjl_ip_est - true_ips).abs().mean() / true_ips.abs().clamp(0.1).mean())
print(f'{'QJL':20} {'1':>5} {qjl_mse:>10.4f} {qjl_ip_err:>10.2%} {qjl.memory_bytes(1):>10} {d*4/qjl.memory_bytes(1):>7.1f}×')

# PolarQuant + TurboQuant
for bits in [2.0, 3.0, 4.0, 8.0]:
    tq       = TurboQuantizer(d, total_bits_per_dim=bits)
    tq_codes = tq.encode(v)
    tq_hat   = tq.decode(tq_codes)
    tq_mse   = float(((v - tq_hat)**2).mean()) / sigma_sq
    tq_ip    = tq.estimate_inner_products(q, tq_codes)
    tq_ip_err = float((tq_ip - true_ips).abs().mean() / true_ips.abs().clamp(0.1).mean())
    mem = tq.memory_bytes(1)
    print(f'{'TurboQuant':20} {bits:>5.1f} {tq_mse:>10.4f} {tq_ip_err:>10.2%} {mem:>10} {d*4/mem:>7.1f}×')

print('-'*70)
for bits in [4.0]:
    theory = theoretical_distortion(sigma_sq, bits) / sigma_sq
    print(f'{'Theory D*(R)':20} {bits:>5.1f} {theory:>10.4f} {"—":>10} {"—":>10} {"—":>8}')
print('='*75)
print('\n✅ All algorithm tests complete! See saved PNG files for figures.')